In [1]:
import json
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# Force Plotly to use the dedicated Google Colab rendering engine
pio.renderers.default = "colab"

# 1. Load the trace map
# INPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/trace_map.json"
INPUT_JSON_PATH = "../data/processed/trace_map.json"
with open(INPUT_JSON_PATH, "r") as f:
    trace_map = json.load(f)

# 2. Prepare the dropdown options
dropdown_options = {}
for article_id, data in trace_map.items():
    snippet = data['text'][:50].replace('\n', ' ') + "..."
    label = f"Article {article_id} - {snippet}"
    dropdown_options[label] = article_id

# 3. Create a sandbox output area to prevent rendering blocks in Colab
output_area = widgets.Output()

# 4. Define the plotting function
def plot_saliency_map(selected_article_id):
    # Clear the previous visualization safely
    output_area.clear_output(wait=True)

    with output_area:
        # Extract data for the selected article
        data = trace_map[selected_article_id]
        scores = data['all_layer_scores']
        n_layers = len(scores)

        # Dynamic Sorting: Find the indices of the top 3 highest scores
        top_3_layers = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]

        # Create the color array (Red for the dynamic top 3 layers, else Blue)
        marker_colors = ['#ef4444' if idx in top_3_layers else '#3b82f6' for idx in range(n_layers)]

        # Generate the Plotly figure
        fig = go.Figure(data=[
            go.Bar(
                x=list(range(n_layers)),
                y=scores,
                marker_color=marker_colors,
                name="Recovery Score"
            )
        ])

        # Update layout with dynamic yaxis auto-scaling
        fig.update_layout(
            title=f"Mechanistic Localization for Article {selected_article_id}",
            xaxis_title="Transformer Layer",
            yaxis_title="Recovery of Target Fact (%)",
            yaxis=dict(autorange=True), # Enforces absolute auto-scaling for variable ranges
            template="plotly_white",
            height=500,
            margin=dict(l=40, r=40, t=60, b=40)
        )

        # Show context metadata above the graph
        print(f"Top 3 Identified Layers (Highest Impact): {top_3_layers}")
        print(f"Original Saved Layers: {data['top_layers']}\n")
        print(f"Target Text Snippet:\n{data['text'][:250]}...\n")

        # Render explicitly using colab protocol
        fig.show(renderer="colab")

# 5. Create the Interactive Widget
article_dropdown = widgets.Dropdown(
    options=dropdown_options,
    description='Select Data:',
    layout={'width': '80%'}
)

# 6. Event observer handler to securely update the canvas
def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        plot_saliency_map(change['new'])

article_dropdown.observe(on_dropdown_change)

# Display the user interface components
print("--- Interactive Saliency Map Explorer ---")
display(article_dropdown, output_area)

# Initialize the notebook with the first plot pre-loaded
initial_key = list(dropdown_options.values())[0]
plot_saliency_map(initial_key)

--- Interactive Saliency Map Explorer ---


Dropdown(description='Select Data:', layout=Layout(width='80%'), options={'Article 0 - Greek Prime Minister Ky…

Output()